# event_log 변환 SQL 생성 (page_view 최근 30일 추가분 전용)

`page_view_logs_recent30d.sql` (generate_page_view_logs_recent30d.ipynb 실행 결과)을 읽어,
`event_log` 테이블에 맞는 INSERT문으로 변환합니다.

## event_log 테이블 컬럼
```
id, user_id, event_name, event_timestamp, product_id, product_name, product_category,
approved_amount, action_type, coupon_code, discount_amount, expiry_date,
search_keyword, page_name, dwell_time, review_rating, earned_points, earn_reason,
login_id, created_at, ad_id, client_uuid
```

## 매핑 규칙 (page_view)
- user_id, login_id, client_uuid, event_timestamp (공통)
- page_name, dwell_time
- 그 외 컬럼은 모두 NULL (id는 SERIAL이므로 INSERT문에 포함하지 않음)

## 사용 방법
1. `page_view_logs_recent30d.sql` 파일을 이 노트북과 같은 디렉토리에 둔다
2. 노트북을 실행하면 `event_log_insert_recent30d.sql`이 생성된다

In [1]:
import re
import json
import os

In [2]:
# ───────────────────────────────────────────
# 설정값
# ───────────────────────────────────────────
SOURCE_SQL_FILE = 'page_view_logs_recent30d.sql'
OUTPUT_SQL_FILE = 'event_log_insert_recent30d.sql'

In [3]:
# event_log 테이블 컬럼 순서 (id 제외)
EVENT_LOG_COLUMNS = [
    'user_id', 'event_name', 'event_timestamp', 'product_id', 'product_name',
    'product_category', 'approved_amount', 'action_type', 'coupon_code',
    'discount_amount', 'expiry_date', 'search_keyword', 'page_name',
    'dwell_time', 'review_rating', 'earned_points', 'earn_reason',
    'login_id', 'created_at', 'ad_id', 'client_uuid'
]

In [4]:
def extract_rows_from_sql(sql_text):
    """
    INSERT INTO ... (history_timestamp, json_log) VALUES
      ('2026-06-01 00:00:01.000000', '{"event_name": ...}'),
      ...
    형식의 SQL 문자열에서 (history_timestamp, json_log) 튜플 리스트를 추출한다.
    json_log 내부의 '' (escaped single quote)는 '로 복원한다.
    """
    pattern = re.compile(
        r"\(\s*'([^']*(?:''[^']*)*)'\s*,\s*'((?:[^']|'')*)'\s*\)",
        re.DOTALL
    )

    rows = []
    for m in pattern.finditer(sql_text):
        history_ts_raw = m.group(1)
        json_log_raw   = m.group(2)

        history_ts = history_ts_raw.replace("''", "'")
        json_log   = json_log_raw.replace("''", "'")

        rows.append((history_ts, json_log))

    return rows

In [5]:
def sql_literal(value):
    """파이썬 값을 SQL 리터럴 문자열로 변환 (None -> NULL, 문자열은 quote+escape)"""
    if value is None:
        return 'NULL'
    if isinstance(value, bool):
        return 'TRUE' if value else 'FALSE'
    if isinstance(value, (int, float)):
        return str(value)
    escaped = str(value).replace("'", "''")
    return f"'{escaped}'"

In [6]:
def map_to_event_log(history_ts, json_log):
    """
    history_timestamp + json_log(dict)를 event_log 테이블 컬럼 dict로 매핑한다.
    이 노트북은 page_view 전용이므로 page_name, dwell_time만 채운다.
    """
    data = json.loads(json_log)

    row = {col: None for col in EVENT_LOG_COLUMNS}

    # ── 공통 ──
    row['user_id']         = data.get('user_id')
    row['event_name']      = data.get('event_name')
    row['event_timestamp'] = data.get('event_timestamp')
    row['login_id']        = data.get('user_login_id')
    row['created_at']      = history_ts
    row['client_uuid']     = data.get('client_uuid')

    # ── page_view 전용 ──
    row['page_name']  = data.get('pageName')
    row['dwell_time'] = data.get('dwellTime')

    return row

In [7]:
if not os.path.exists(SOURCE_SQL_FILE):
    raise FileNotFoundError(f'{SOURCE_SQL_FILE} 이 없습니다. generate_page_view_logs_recent30d.ipynb를 먼저 실행해 주세요.')

with open(SOURCE_SQL_FILE, 'r', encoding='utf-8') as f:
    sql_text = f.read()

raw_rows = extract_rows_from_sql(sql_text)
all_event_rows = [map_to_event_log(history_ts, json_log) for history_ts, json_log in raw_rows]

print(f'✅ {SOURCE_SQL_FILE} → {len(all_event_rows)}건 변환')

✅ page_view_logs_recent30d.sql → 1000건 변환


In [8]:
# event_log INSERT SQL 생성
columns_str = ', '.join(EVENT_LOG_COLUMNS)

lines  = [f'INSERT INTO event_log ({columns_str}) VALUES']
values = []

for row in all_event_rows:
    literals = [sql_literal(row[col]) for col in EVENT_LOG_COLUMNS]
    values.append('  (' + ', '.join(literals) + ')')

lines.append(',\n'.join(values) + ';')
event_log_sql = '\n'.join(lines)

with open(OUTPUT_SQL_FILE, 'w', encoding='utf-8') as f:
    f.write(event_log_sql)

print(f'✅ {len(all_event_rows)}건 event_log INSERT SQL 생성 완료 → {OUTPUT_SQL_FILE}')

✅ 1000건 event_log INSERT SQL 생성 완료 → event_log_insert_recent30d.sql


In [9]:
# ── 미리보기 ──
print('=== EVENT_LOG INSERT SQL (앞 1000자) ===')
print(event_log_sql[:1000])

=== EVENT_LOG INSERT SQL (앞 1000자) ===
INSERT INTO event_log (user_id, event_name, event_timestamp, product_id, product_name, product_category, approved_amount, action_type, coupon_code, discount_amount, expiry_date, search_keyword, page_name, dwell_time, review_rating, earned_points, earn_reason, login_id, created_at, ad_id, client_uuid) VALUES
  (67, 'page_view', '2026-06-14T16:32:37.000+09:00', NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, '마이페이지', 29, NULL, NULL, NULL, 'user0067', '2026-06-14 16:32:38.000000', NULL, '5390d67d-6938-4fa1-aec9-74a588fd218c'),
  (39, 'page_view', '2026-05-31T20:08:46.000+09:00', NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, '주문완료', 5, NULL, NULL, NULL, 'user0039', '2026-05-31 20:08:47.000000', NULL, 'eff8b058-689a-41b5-9e3e-8f06036d018e'),
  (NULL, 'page_view', '2026-06-02T07:47:08.000+09:00', NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, '상품목록', 18, NULL, NULL, NULL, NULL, '2026-06-02 07:47:09.000000', NULL, 'c664c494-d765-46c7